# components.collection

> Main entry point for rendering a virtual collection.

In [ ]:
#| default_exp components.collection

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Callable, List, Optional

from fasthtml.common import Div, Hidden

from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.utilities.sizing import w, min_h
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, flex_direction, flex, grow
from cjm_fasthtml_tailwind.utilities.interactivity import touch
from cjm_fasthtml_tailwind.utilities.tables import table_display, border_collapse
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_fasthtml_virtual_collection.core.models import (
    VirtualCollectionConfig, VirtualCollectionState, VirtualCollectionUrls,
)
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds
from cjm_fasthtml_virtual_collection.components.table import render_header_row, render_table_rows
from cjm_fasthtml_virtual_collection.components.footer import render_footer
from cjm_fasthtml_virtual_collection.components.scrollbar import render_scrollbar

In [ ]:
#| export
def render_virtual_collection(
    items: list,                                # Full item list
    config: VirtualCollectionConfig,             # Collection config
    state: VirtualCollectionState,               # Collection state
    ids: VirtualCollectionHtmlIds,               # HTML IDs
    urls: VirtualCollectionUrls,                 # URL bundle
    render_cell: Optional[Callable] = None,      # Table layout cell render callback
    render_item: Optional[Callable] = None,      # Grid layout item render callback
    render_empty: Optional[Callable] = None,     # Empty state callback: () -> FT component
) -> Div:  # Complete collection element
    """Render a complete virtual collection with wrapper, table, scrollbar, and footer."""
    children = []

    if config.layout == "table":
        assert render_cell is not None, "render_cell required for table layout"

        # Empty state: replace table body with consumer-provided component
        if state.total_items == 0 and render_empty is not None:
            table = Div(
                Div(
                    render_header_row(config, ids, state=state, sort_url=urls.sort),
                    cls=combine_classes(table_display.header_group),
                ),
                render_empty(),
                id=ids.table,
                cls=combine_classes(table_display, w.full, border_collapse.collapse),
            )
        else:
            # CSS Table: header group + body group
            table = Div(
                Div(
                    render_header_row(config, ids, state=state, sort_url=urls.sort),
                    cls=combine_classes(table_display.header_group),
                ),
                render_table_rows(items, config, state, ids, render_cell, focus_url=urls.focus_row),
                id=ids.table,
                cls=combine_classes(table_display, w.full, border_collapse.collapse),
            )

        # Wrapper (viewport-fit target) + scrollbar side-by-side
        # touch-pan-x: allow native horizontal pan for wide tables,
        # suppress vertical (handled by discrete navigation)
        # min-h-0: allow shrinking below content height in flex parents
        wrapper = Div(
            table,
            id=ids.wrapper,
            cls=combine_classes(flex(1), min_h._0, overflow.y.hidden, touch.pan_x),
        )

        if config.show_scrollbar and state.total_items > state.visible_rows:
            scrollbar = render_scrollbar(state, config, ids)
            body = Div(wrapper, scrollbar, cls=combine_classes(flex_display, grow(), min_h._0))
        else:
            body = wrapper

        children.append(body)

    elif config.layout == "grid":
        raise NotImplementedError("Grid layout will be added in Phase 14")
    else:
        raise ValueError(f"Unknown layout: {config.layout}")

    # Footer
    children.append(render_footer(state, ids))

    # Hidden input for JS thumb positioning (OOB-updated on navigation)
    children.append(Hidden(
        value=str(state.window_start),
        id=ids.window_start_input,
        name="window_start",
    ))

    # grow + min-h-0: fill flex parent without exceeding it
    return Div(
        *children,
        id=ids.collection,
        cls=combine_classes(flex_display, flex_direction.col, grow(), min_h._0, touch.pan_x),
    )

## Tests

In [ ]:
from dataclasses import dataclass
from fasthtml.common import to_xml, Span
from cjm_fasthtml_virtual_collection.core.models import (
    VirtualCollectionConfig, VirtualCollectionState, ColumnDef, VirtualCollectionUrls,
)
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds

@dataclass
class Item:
    name: str
    size: str

test_items = [Item(f"file_{i}.txt", f"{i}KB") for i in range(20)]

def test_render_cell(item, ctx):
    if ctx.column.key == "name": return Span(item.name)
    if ctx.column.key == "size": return Span(item.size)

columns = (
    ColumnDef(key="name", header="Name"),
    ColumnDef(key="size", header="Size"),
)
config = VirtualCollectionConfig(prefix="tc", columns=columns)
state = VirtualCollectionState(total_items=20, visible_rows=5)
ids = VirtualCollectionHtmlIds(prefix=config.prefix)
urls = VirtualCollectionUrls()

In [ ]:
# Test full collection render
collection = render_virtual_collection(
    items=test_items, config=config, state=state,
    ids=ids, urls=urls, render_cell=test_render_cell,
)
html = to_xml(collection)

# Outer container with flex-child classes
assert 'id="tc-collection"' in html
assert 'grow-1' in html or 'grow' in html
assert 'min-h-0' in html

# Wrapper (viewport-fit target)
assert 'id="tc-wrapper"' in html
assert 'overflow-y-hidden' in html

# Table
assert 'id="tc-table"' in html

# Header (inside table-header-group)
assert 'id="tc-header"' in html
assert 'table-header-group' in html
assert 'Name' in html
assert 'Size' in html

# Rows (table-row-group)
assert 'id="tc-rows"' in html
assert 'table-row-group' in html

# Visible rows (0-4, visible_rows=5)
assert 'id="tc-row-0"' in html
assert 'id="tc-row-4"' in html
assert 'id="tc-row-5"' not in html

# Slots with display:contents
assert 'id="tc-slot-0"' in html
assert 'contents' in html

# Cell IDs
assert 'id="tc-row-0-col-name"' in html
assert 'id="tc-row-0-col-size"' in html

# Scrollbar (20 items > 5 visible)
assert 'id="tc-scrollbar-track"' in html
assert 'id="tc-scrollbar-thumb"' in html

# Footer
assert 'id="tc-footer"' in html
assert 'Showing 1-5 of 20 items' in html

# Cell content
assert 'file_0.txt' in html
assert 'file_4.txt' in html
assert 'file_5.txt' not in html

print("Full collection render test passed")

In [ ]:
# Test with window_start offset
state2 = VirtualCollectionState(total_items=20, visible_rows=5, window_start=10)
collection2 = render_virtual_collection(
    items=test_items, config=config, state=state2,
    ids=ids, urls=urls, render_cell=test_render_cell,
)
html2 = to_xml(collection2)

assert 'id="tc-row-10"' in html2
assert 'id="tc-row-14"' in html2
assert 'id="tc-row-9"' not in html2
assert 'id="tc-row-15"' not in html2
assert 'Showing 11-15 of 20 items' in html2
assert 'file_10.txt' in html2

print("Offset window render test passed")

Offset window render test passed


In [ ]:
# Test empty state with render_empty callback
from fasthtml.common import P

def my_empty_state():
    return P("No files found", id="empty-msg")

empty_state = VirtualCollectionState(total_items=0, visible_rows=5)
collection_empty = render_virtual_collection(
    items=[], config=config, state=empty_state,
    ids=ids, urls=urls, render_cell=test_render_cell,
    render_empty=my_empty_state,
)
html_empty = to_xml(collection_empty)

# Empty state component is rendered
assert 'No files found' in html_empty
assert 'id="empty-msg"' in html_empty

# Header is still present
assert 'id="tc-header"' in html_empty
assert 'Name' in html_empty

# No data rows
assert 'id="tc-row-0"' not in html_empty

# No scrollbar (0 items)
assert 'id="tc-scrollbar-track"' not in html_empty

# Footer shows "No items"
assert 'No items' in html_empty

print("Empty state render test passed")

Empty state render test passed


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()